# Demo · Agenti: RAG classico vs Foundry IQ

**Use case** (route `/demo/agenti` della chat): la stessa domanda è posta a tre agenti del progetto Foundry.

| Agente | Strategia |
| --- | --- |
| `wpc-rag-classico-rapido` | Tool indipendenti: indice manuali keyword top 3, MCP ricambi, web filtrato. |
| `wpc-rag-classico-accurato` | Stesse fonti con indice semantico top 10, tutti i tool MCP e prompt di orchestrazione. |
| `wpc-technician-assistant` | Un solo tool: la knowledge base Foundry IQ pianifica le query su tutte le fonti. |

Inoltre `wpc-magazzino-assistant` mostra un agente che usa **direttamente** il server MCP ricambi.

Il notebook legge l'ambiente azd selezionato oppure gli output ARM e confronta tool invocati, chiamate, token e tempi. Gli agenti usano l'identità del progetto: le ACL dell'utente non si applicano.

In [ ]:
CONFIG_SOURCE = "azd"             # "azd" oppure "arm"
AZD_ENVIRONMENT_NAME = ""          # vuoto = ambiente azd attualmente selezionato
SUBSCRIPTION_ID = ""               # opzionale; con azd deve coincidere con l'ambiente
ENVIRONMENT_NAME = "wpc-demo"      # usato solo per il deployment ARM
DEPLOYMENT_NAME = f"wpc-foundryiq-{ENVIRONMENT_NAME}"

CONFRONTO = ["wpc-rag-classico-rapido", "wpc-rag-classico-accurato", "wpc-technician-assistant"]
DOMANDE = [
    "Devo cambiare le membrane filtranti della Graphene Vapor Reclaim Tank: ogni quanto va fatto, quante GRT-002 abbiamo a magazzino e come vanno smaltite quelle esauste?",
    "L'Edge Control Node P4311 non si accende: come verifico l'alimentatore AD-60W-24, ne abbiamo a magazzino o un'alternativa, e quali modelli equivalenti vendono i distributori online?",
]
MAGAZZINO = "wpc-magazzino-assistant"
DOMANDA_MAGAZZINO = "Quali ricambi ad alta criticità sono sotto scorta e in quale magazzino?"

In [ ]:
import json
import re
import shutil
import subprocess
import sys
import time
from collections.abc import Mapping
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: servono infra/, data/, agents/ e le app.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from notebook_env import load_azd_env
AZ = shutil.which("az")
if AZ is None:
    raise RuntimeError("Azure CLI non trovata nel PATH: installala, esegui 'az login' e riavvia il kernel.")


def az(*args, output=True, check=True):
    """Esegue Azure CLI. Con output=True restituisce il JSON decodificato; con check=False None in caso di errore."""
    command = [AZ, *args, "--only-show-errors"] + (["--output", "json"] if output else [])
    proc = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        if check:
            raise RuntimeError(f"az {' '.join(args[:3])} fallito: {proc.stderr.strip()[-2000:]}")
        return None
    text = proc.stdout.strip()
    return (json.loads(text) if text else None) if output else text


def load_azd_values(environment_name):
    return load_azd_env(environment_name, root=ROOT, override=True)


def load_outputs(deployment_name, subscription_id):
    deployment = az("deployment", "sub", "show", "--name", deployment_name, "--subscription", subscription_id)
    if not deployment:
        raise RuntimeError(f"Deployment ARM '{deployment_name}' non trovato nella subscription selezionata.")
    state = deployment["properties"].get("provisioningState")
    if state != "Succeeded":
        raise RuntimeError(f"Deployment '{deployment_name}' in stato {state}: correggi e riesegui 01-infrastruttura.ipynb.")
    return {name.upper(): item["value"] for name, item in (deployment["properties"].get("outputs") or {}).items()}


def load_configuration(required):
    if CONFIG_SOURCE == "azd":
        out = load_azd_values(AZD_ENVIRONMENT_NAME)
        selected = out.get("AZURE_SUBSCRIPTION_ID")
        if not isinstance(selected, str) or not selected.strip():
            raise RuntimeError("AZURE_SUBSCRIPTION_ID mancante nell'ambiente azd selezionato.")
        if SUBSCRIPTION_ID and SUBSCRIPTION_ID.casefold() != selected.casefold():
            raise RuntimeError("SUBSCRIPTION_ID non corrisponde alla subscription dell'ambiente azd.")
        account = az("account", "show", "--subscription", selected)
        if account["id"].casefold() != selected.casefold():
            raise RuntimeError("Azure CLI ha restituito una subscription diversa da quella dell'ambiente azd.")
        source = f"ambiente azd {out.get('AZURE_ENV_NAME') or AZD_ENVIRONMENT_NAME or '(corrente)'}"
    elif CONFIG_SOURCE == "arm":
        account = az("account", "show", *(["--subscription", SUBSCRIPTION_ID] if SUBSCRIPTION_ID else []))
        out = load_outputs(DEPLOYMENT_NAME, account["id"])
        source = f"deployment ARM {DEPLOYMENT_NAME}"
    else:
        raise ValueError("CONFIG_SOURCE deve essere 'azd' oppure 'arm'.")
    missing = [key for key in ["AZURE_TENANT_ID", "AZURE_RESOURCE_GROUP", *required]
               if not isinstance(out.get(key), str) or not out[key].strip()]
    if missing:
        raise RuntimeError(f"Valori mancanti in {source}: {', '.join(missing)}.")
    if out["AZURE_TENANT_ID"].casefold() != account["tenantId"].casefold():
        raise RuntimeError("Il tenant della configurazione non corrisponde alla subscription selezionata in Azure CLI.")
    return out, account, source


out, account, source = load_configuration(["FOUNDRYIQ_DEMO_PROJECT_ENDPOINT"])
SUBSCRIPTION_ID, TENANT_ID = account["id"], account["tenantId"]
RESOURCE_GROUP = out["AZURE_RESOURCE_GROUP"]
print(f"Subscription: {account['name']} ({SUBSCRIPTION_ID})")
print(f"Configurazione: {source} -> resource group {RESOURCE_GROUP}")

import pandas as pd
import requests
from azure.identity import AzureCliCredential
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
credential = AzureCliCredential(subscription=SUBSCRIPTION_ID)
SEARCH_SCOPE = "https://search.azure.com/.default"
checks = []


def record(test, status, detail=""):
    """Esito di un controllo: PASS, WARN (non bloccante), FAIL o SKIP."""
    checks.append({"controllo": test, "esito": status, "dettaglio": detail})
    print(f"[{status}] {test}" + (f" - {detail}" if detail else ""))

from azure.ai.projects import AIProjectClient

project = AIProjectClient(endpoint=out["FOUNDRYIQ_DEMO_PROJECT_ENDPOINT"], credential=credential)
openai_client = project.get_openai_client()


def as_dict(item):
    if item is None:
        return {}
    if isinstance(item, Mapping):
        return dict(item)
    dump = getattr(item, "model_dump", None)
    if callable(dump):
        result = dump()
        if isinstance(result, Mapping):
            return dict(result)
    raise TypeError(f"Tipo non supportato dall'SDK: {type(item).__name__}")

In [ ]:
TOOL_LABELS = {"mcp_call": "MCP", "azure_ai_search_call": "Azure AI Search", "web_search_call": "Web", "file_search_call": "File"}


def summarize_response(response):
    """Estrae risposta, tool invocati, citazioni e token da una risposta della Responses API."""
    data = as_dict(response)
    answer, calls, citations = [], [], {}
    for item in data.get("output") or []:
        item = as_dict(item)
        kind = item.get("type") or ""
        if kind == "message":
            for content in item.get("content") or []:
                content = as_dict(content)
                answer.append(content.get("text") or "")
                for note in content.get("annotations") or []:
                    note = as_dict(note)
                    key = note.get("url") or note.get("title") or note.get("file_id")
                    if key:
                        citations[key] = note.get("title") or key
        elif kind.endswith("_call"):
            calls.append({"tipo": TOOL_LABELS.get(kind, kind), "tool": item.get("name") or item.get("server_label") or kind,
                          "server": item.get("server_label"), "stato": item.get("status"), "errore": item.get("error")})
    usage = as_dict(data.get("usage"))
    return {"risposta": "\n".join(a for a in answer if a), "chiamate": calls, "citazioni": list(citations.values()),
            "token input": usage.get("input_tokens"), "token output": usage.get("output_tokens"), "token totali": usage.get("total_tokens")}

## 1 · Agenti nel progetto Foundry

In [ ]:
inventory = []
for name in CONFRONTO + [MAGAZZINO]:
    try:
        agent = as_dict(project.agents.get(name))
        latest = as_dict(as_dict(agent.get("versions")).get("latest"))
        definition = as_dict(latest.get("definition"))
    except Exception as exc:  # noqa: BLE001 - un agente mancante diventa un FAIL leggibile
        record(f"Agente presente · {name}", "FAIL", f"{type(exc).__name__}: {str(exc)[:200]}")
        continue
    if not latest.get("version") or not definition:
        record(f"Agente presente · {name}", "FAIL", "versione o definizione non disponibile")
        continue
    tools = definition.get("tools") or []
    inventory.append({"agente": name, "versione": latest.get("version"), "modello": definition.get("model"),
                      "tool configurati": ", ".join(t.get("server_label") or t.get("type") for t in tools)})
    record(f"Agente presente · {name}", "PASS", f"versione {latest.get('version')}, {len(tools)} tool")
display(pd.DataFrame(inventory))

## 2 · Stessa domanda, tre strategie

In [ ]:
def ask(agent, question):
    started = time.perf_counter()
    response = openai_client.responses.create(input=[{"role": "user", "content": question}],
                                              extra_body={"agent_reference": {"name": agent, "type": "agent_reference"}})
    summary = summarize_response(response)
    summary.update({"agente": agent, "stato": response.status, "secondi": round(time.perf_counter() - started, 1)})
    return summary


runs = []
for question in DOMANDE:
    display(Markdown(f"**Domanda:** {question}"))
    rows = []
    for agent in CONFRONTO:
        try:
            run = ask(agent, question)
        except Exception as exc:  # noqa: BLE001
            record(f"Risposta · {agent}", "FAIL", str(exc)[:200])
            continue
        run["domanda"] = question
        runs.append(run)
        by_type = {}
        for call in run["chiamate"]:
            by_type[call["tipo"]] = by_type.get(call["tipo"], 0) + 1
        rows.append({"agente": agent, "tool (chiamate)": ", ".join(f"{k} ×{v}" for k, v in by_type.items()) or "nessuno",
                     "chiamate": len(run["chiamate"]), "errori tool": sum(1 for c in run["chiamate"] if c["errore"]),
                     "citazioni": len(run["citazioni"]), "token totali": run["token totali"], "secondi": run["secondi"]})
        record(f"Risposta · {agent}", "PASS" if run["stato"] == "completed" and run["risposta"].strip() else "FAIL",
               f"{len(run['chiamate'])} chiamate tool, {run['token totali']} token")
    display(pd.DataFrame(rows))

## 3 · Dettaglio delle chiamate e risposte

In [ ]:
for run in runs:
    display(Markdown(f"### {run['agente']}\n*{run['domanda'][:90]}…*"))
    if run["chiamate"]:
        display(pd.DataFrame(run["chiamate"]))
    display(Markdown(run["risposta"][:3000] or "_Nessuna risposta_"))
fiq = [r for r in runs if r["agente"] == "wpc-technician-assistant"]
record("Foundry IQ usa la knowledge base", "PASS" if fiq and all(any(c["tipo"] == "MCP" for c in r["chiamate"]) for r in fiq) else "WARN",
       "tool knowledge_base_retrieve (MCP) invocato")

## 4 · Agente magazzino con MCP diretto

In [ ]:
try:
    run = ask(MAGAZZINO, DOMANDA_MAGAZZINO)
    display(pd.DataFrame(run["chiamate"]) if run["chiamate"] else Markdown("_Nessun tool invocato_"))
    display(Markdown(run["risposta"][:3000]))
    record("Magazzino · tool MCP ricambi", "PASS" if any(c["tipo"] == "MCP" and not c["errore"] for c in run["chiamate"]) else "FAIL",
           ", ".join(c["tool"] for c in run["chiamate"]) or "nessuna chiamata")
except Exception as exc:  # noqa: BLE001
    record("Magazzino · tool MCP ricambi", "FAIL", str(exc)[:200])

In [ ]:
openai_client.close()
project.close()
credential.close()
display(pd.DataFrame(checks))
failed = [c["controllo"] for c in checks if c["esito"] == "FAIL"]
if failed:
    raise AssertionError("Controlli falliti: " + ", ".join(failed))
print("Demo agenti verificata.")